In [1]:
import sys
sys.path.append("..")  # para poder importar src desde la carpeta notebooks

import pandas as pd
from src.nlp.normalizer import Normalizador

normalizador = Normalizador(quitar_tildes=True)

ejemplos = [
    ("Pérdida de la ñ (corregido)", "El año pasado comí piña"),
    ("Letras repetidas",            "RIQUÍSIMOOOO, muuuuy bueeeno"),
    ("Letras dobles válidas",       "La llama y el carro"),
    ("URL",                         "Vean https://tripadvisor.com"),
    ("Emojis",                      "Delicioso 😋👍 volveremos"),
    ("Puntuación",                  "¡¡Excelente servicio!! ¿Volvería? Sí."),
    ("Calificación (se conserva)",  "Todo perfecto, 10/10"),
    ("Precio (se conserva)",        "Pagamos $150, algo caro"),
]

tabla = pd.DataFrame(
    [(caso, texto, normalizador.normalizar(texto)) for caso, texto in ejemplos],
    columns=["Caso", "Antes", "Después"],
)
pd.set_option("display.max_colwidth", None)
tabla

,Caso,Antes,Después
0,Pérdida de la ñ (corregido),El año pasado comí piña,el año pasado comi piña
1,Letras repetidas,"RIQUÍSIMOOOO, muuuuy bueeeno",riquisimo muy bueno
2,Letras dobles válidas,La llama y el carro,la llama y el carro
3,URL,Vean https://tripadvisor.com,vean
4,Emojis,Delicioso 😋👍 volveremos,delicioso volveremos
5,Puntuación,¡¡Excelente servicio!! ¿Volvería? Sí.,excelente servicio volveria si
6,Calificación (se conserva),"Todo perfecto, 10/10",todo perfecto 10/10
7,Precio (se conserva),"Pagamos $150, algo caro",pagamos $150 algo caro


In [2]:
df = pd.read_csv("../data/raw/restaurantes.csv")

muestra = df.sample(5, random_state=42)

tabla_reales = pd.DataFrame({
    "Sentimiento": muestra["sentimiento"].values,
    "Antes": muestra["texto"].values,
    "Después": [normalizador.normalizar(t) for t in muestra["texto"]],
})
tabla_reales

,Sentimiento,Antes,Después
0,positivo,Excelente comida. muy buena calidad en la comida. Los platos son super ricos y a precios accesibles. Tiene una hermosa vista.,excelente comida muy buena calidad en la comida los platos son super ricos y a precios accesibles tiene una hermosa vista
1,positivo,"Buena comida, cocina saludable, fresca. Un muy buen menú con interesante, saludable, frescas opciones. La salsa veges fresca con pollo es magnífico, una comida abundante y bastante mediano-caliente. colorado El pescado estaba muy fresco. Vino era bueno y servida con generosidad por copa.",buena comida cocina saludable fresca un muy buen menu con interesante saludable frescas opciones la salsa veges fresca con pollo es magnifico una comida abundante y bastante mediano caliente colorado el pescado estaba muy fresco vino era bueno y servida con generosidad por copa
2,positivo,"El desayuno para no perdérselo. Desayunamos aquí dos veces. Las tortitas rollitos eran increíbles. El Coconut tostadas, mismo, increíble. La guarnición: fruta, tan bueno, buen toque. Café de autoservicio, genial, desde un amante del café. interior precioso! El personal es muy agradable. Mi hija se inhalar los panqueques rollitos cuando...volvamos en julio!!Más",el desayuno para no perderselo desayunamos aqui dos veces las tortitas rollitos eran increibles el coconut tostadas mismo increible la guarnicion fruta tan bueno buen toque cafe de autoservicio genial desde un amante del cafe interior precioso el personal es muy agradable mi hija se inhalar los panqueques rollitos cuando volvamos en julio mas
3,positivo,"EXCELENTE OPCIÓN EN VARIEDAD DE COMIDA A BUEN PRECIO.. Es un excelente lugar para comer y cenar, cuenta con entretenimiento con show artístico o música en vivo, recomendable visitar despues de ir a la zona arqueológica.\n\nManejan buenos precios, solo que aceptan puro efectivo, no hay cobro con terminales.",excelente opcion en variedad de comida a buen precio es un excelente lugar para comer y cenar cuenta con entretenimiento con show artistico o musica en vivo recomendable visitar despues de ir a la zona arqueologica manejan buenos precios solo que aceptan puro efectivo no hay cobro con terminales
4,positivo,"Delicioso y con excelente atención de Edwin. Estuvo delicioso con increíbles opciones veganas pero quiero recalcar la excelente atención de nuestro mesero, Edwin. Súper amable, atento y dispuesto a responder todas nuestras preguntas",delicioso y con excelente atencion de edwin estuvo delicioso con increibles opciones veganas pero quiero recalcar la excelente atencion de nuestro mesero edwin super amable atento y dispuesto a responder todas nuestras preguntas


## Normalización: qué se elimina, qué se conserva y por qué

| Paso | Decisión | Justificación |
|---|---|---|
| Minúsculas | Se aplica | "Excelente" y "excelente" deben ser la misma palabra. |
| URLs | Se eliminan | No aportan sentimiento; solo 22 reseñas del dataset las tienen. |
| Emojis | Se eliminan | El dataset no contiene ningún emoji, así que el modelo no los conoce. |
| Letras repetidas | 3 o más → 1 | "riquisimoooo" y "riquisimo" deben ser la misma palabra. Las dobles se conservan porque son válidas en español ("llama", "carro"). Solo se aplica a letras para no alterar números ("1000"). |
| Tildes | Se eliminan, **conservando la ñ** | Muchas reseñas se escriben sin tildes: "atención" aparece 13.294 veces y "atencion" 1.309. Sin quitarlas, serían palabras distintas. La ñ se conserva porque cambia el significado ("año" ≠ "ano"). |
| Números | Se conservan | Aportan información como calificaciones y precios. |
| Puntuación | Se elimina, salvo dos casos | Los signos no aportan sentimiento. Se conservan "10/10" (calificación, 193 reseñas) y "$150" (precio, 989 reseñas) como casos del dominio para el tokenizador. |

**Orden de los pasos:** las URLs se quitan antes de reducir letras repetidas (si no, "www" se convierte en "w"), y la puntuación se quita después de los números (para no dejar "/" o "$" sueltos).